# E-Commerce Sales & Customer Analytics
**SQL and Python-based analysis of sales, products, customers, and business performance**

This notebook walks through a complete analysis: raw data, cleaning, a SQLite database, SQL queries, Pandas/NumPy analysis, charts, and business insights.

> **Note:** the dataset is **synthetic**. It is generated by `src/generate_data.py`, so the patterns in it reflect how the data was generated. The goal of the project is to demonstrate the analysis workflow.

**Pipeline:** Raw data → Cleaning → SQLite database → SQL analysis → Pandas / NumPy → Visualizations → Insights → Recommendations

**Contents:** 1. Load raw data · 2. Inspect data quality · 3. Clean the data · 4. Load into SQLite · 5. SQL analysis · 6. Pandas and NumPy analysis · 7. Visualizations · 8. Insights and recommendations

In [1]:
import sqlite3
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", None)

print("Pandas:", pd.__version__)
print("NumPy :", np.__version__)

Pandas: 3.0.6
NumPy : 2.5.3


## 1. Load the raw data
If the raw CSV does not exist yet, the next cell creates it by running `src/generate_data.py`.

In [2]:
RAW_FILE = Path("../data/raw/ecommerce_raw.csv")

if not RAW_FILE.exists():
    %run ../src/generate_data.py

df = pd.read_csv(RAW_FILE)
print("Rows, columns:", df.shape)
df.head()

Exception: File `'../src/generate_data.py'` not found.

In [ ]:
df.info()

## 2. Inspect data quality (before cleaning)
We look for missing values, duplicate rows, and impossible values. We show each problem *before* fixing it.

In [ ]:
# Missing values per column
print(df.isna().sum())

In [ ]:
# Duplicate rows (an order cannot contain the same product twice)
print("Duplicate rows:", df.duplicated().sum())
df[df.duplicated(keep=False)].sort_values("OrderID").head(6)

In [ ]:
# Impossible values
print("Rows with Quantity <= 0:", (df["Quantity"] <= 0).sum())
print("Rows with UnitPrice <= 0:", (df["UnitPrice"] <= 0).sum())
df[df["Quantity"] <= 0].head()

In [ ]:
# Summary statistics: look at the min row
df[["Quantity", "UnitPrice"]].describe()

In [ ]:
# Date range and type (dates are still plain text)
print("Earliest date:", df["OrderDate"].min())
print("Latest date:", df["OrderDate"].max())
print("Date column type:", df["OrderDate"].dtype)

## 3. Clean the data
We clean a **copy** (`df_clean`) and keep the original `df` unchanged.

Decisions:
- **Duplicates:** removed, because they double-count sales.
- **Quantity or price of 0 or below:** removed, because the true value cannot be recovered.
- **Missing country:** filled from the same customer's other rows. If no record has a country, the row is labelled `Unknown` (the sale is real, so it is kept).

In [ ]:
df_clean = df.copy()
print("Starting rows:", len(df_clean))

# 1. Duplicates
rows_before = len(df_clean)
df_clean = df_clean.drop_duplicates()
print("Duplicates removed:", rows_before - len(df_clean))

# 2. Impossible quantities and prices
is_valid = (df_clean["Quantity"] > 0) & (df_clean["UnitPrice"] > 0)
print("Invalid rows removed:", (~is_valid).sum())
df_clean = df_clean[is_valid].copy()

# 3. Dates: text -> real dates
df_clean["OrderDate"] = pd.to_datetime(df_clean["OrderDate"])

# 4. Missing country: copy it from the same customer's other rows
missing_before = df_clean["Country"].isna().sum()
country_by_customer = df_clean.groupby("CustomerID")["Country"].transform("first")
df_clean["Country"] = df_clean["Country"].fillna(country_by_customer)
print("Missing country before:", missing_before, "| after first fix:", df_clean["Country"].isna().sum())

# 4b. Still missing: look in the ORIGINAL data, then label what is left as Unknown
customer_country = df.groupby("CustomerID")["Country"].first()
still_missing = df_clean["Country"].isna()
df_clean.loc[still_missing, "Country"] = df_clean.loc[still_missing, "CustomerID"].map(customer_country)
df_clean["Country"] = df_clean["Country"].fillna("Unknown")
print("Rows labelled Unknown:", (df_clean["Country"] == "Unknown").sum())

# 5. Calculated columns
df_clean["Revenue"] = (df_clean["Quantity"] * df_clean["UnitPrice"]).round(2)
df_clean["Year"] = df_clean["OrderDate"].dt.year
df_clean["Month"] = df_clean["OrderDate"].dt.month
df_clean["YearMonth"] = df_clean["OrderDate"].dt.to_period("M").astype(str)

In [ ]:
# Before vs after
print("Rows before:", len(df), "| Rows after:", len(df_clean))
print("Missing Country:", df_clean["Country"].isna().sum())
print("Duplicates:", df_clean.duplicated().sum())
print("Min Quantity:", df_clean["Quantity"].min())
print("Min UnitPrice:", df_clean["UnitPrice"].min())
df_clean.head()

In [ ]:
# Save the cleaned data
Path("../data/processed").mkdir(parents=True, exist_ok=True)
df_clean.to_csv("../data/processed/ecommerce_clean.csv", index=False)
print("Saved cleaned file.")

## 4. Load the cleaned data into SQLite
`src/load_data.py` splits the clean table into four related tables (`customers`, `products`, `orders`, `order_items`) using `sql/database_setup.sql`. It also checks that total revenue in SQL matches Pandas.

In [ ]:
%run ../src/load_data.py

## 5. SQL analysis
All queries are stored in `sql/analysis_queries.sql` (numbered 1 to 20). The notebook reads that file and runs the query you ask for, so the SQL lives in one place.

In [ ]:
connection = sqlite3.connect("../data/processed/ecommerce.db")

def run_query(sql):
    """Run a SQL query and return the result as a Pandas table."""
    return pd.read_sql(sql, connection)

# Read the SQL file, remove comment lines, and split it into separate queries
sql_text = Path("../sql/analysis_queries.sql").read_text(encoding="utf-8")
lines = [line for line in sql_text.splitlines() if not line.strip().startswith("--")]
queries = [q.strip() for q in "\n".join(lines).split(";") if q.strip()]

def sql_query(number):
    """Run query number N (1 to 20) from analysis_queries.sql."""
    return run_query(queries[number - 1])

print(len(queries), "queries loaded")

In [ ]:
# Query 6: KPI summary
sql_query(6)

In [ ]:
# Query 8: revenue by category
sql_query(8)

In [ ]:
# Query 9: top 10 products by revenue
sql_query(9)

In [ ]:
# Query 10: monthly revenue
sql_query(10)

In [ ]:
# Query 11: revenue by country
sql_query(11)

In [ ]:
# Query 12: top 10 customers
sql_query(12)

In [ ]:
# Query 14: revenue by price tier (CASE WHEN)
sql_query(14)

In [ ]:
# Query 16: customers with no orders in 2024 (LEFT JOIN)
sql_query(16).head()

In [ ]:
# Query 18: customer segments (CTEs + window function)
sql_query(18)

In [ ]:
# Query 19: high units sold but low revenue (RANK window function)
sql_query(19)

In [ ]:
# Query 20: month-over-month change (LAG window function)
sql_query(20)

## 6. Analysis with Pandas and NumPy
We load SQL results into Pandas and calculate KPIs, shares, and customer statistics.

In [ ]:
# SQL results as Pandas tables
category_df = sql_query(8)
top_products_df = sql_query(9)
monthly_df = sql_query(10)
country_df = sql_query(11)

# Revenue of ALL customers (query 12 only returns the top 10)
customer_df = run_query("""
    SELECT c.customer_id, c.customer_name,
           COUNT(DISTINCT o.order_id) AS orders,
           ROUND(SUM(oi.quantity * p.unit_price), 2) AS revenue
    FROM order_items AS oi
    INNER JOIN orders    AS o ON oi.order_id   = o.order_id
    INNER JOIN customers AS c ON o.customer_id = c.customer_id
    INNER JOIN products  AS p ON oi.product_id = p.product_id
    GROUP BY c.customer_id, c.customer_name
    ORDER BY revenue DESC
""")

print("category_df :", category_df.shape)
print("monthly_df  :", monthly_df.shape)
print("country_df  :", country_df.shape)
print("customer_df :", customer_df.shape)

In [ ]:
# KPI summary
total_revenue = df_clean["Revenue"].sum()
total_orders = df_clean["OrderID"].nunique()

kpi = pd.DataFrame({
    "KPI": ["Total Revenue", "Total Orders", "Total Customers",
            "Total Products", "Average Order Value", "Total Units Sold"],
    "Value": [f"{total_revenue:,.2f}",
              f"{total_orders:,}",
              f"{df_clean['CustomerID'].nunique():,}",
              f"{df_clean['ProductID'].nunique():,}",
              f"{total_revenue / total_orders:,.2f}",
              f"{df_clean['Quantity'].sum():,}"]
})
kpi

In [ ]:
# Category shares of revenue and units
category_df["revenue_share_pct"] = (category_df["revenue"] / category_df["revenue"].sum() * 100).round(1)
category_df["units_share_pct"] = (category_df["units_sold"] / category_df["units_sold"].sum() * 100).round(1)
category_df

In [ ]:
# Customer statistics with NumPy
revenues = customer_df["revenue"].to_numpy()

print("Average revenue per customer:", round(revenues.mean(), 2))
print("Median revenue per customer :", round(np.median(revenues), 2))

sorted_revenues = np.sort(revenues)[::-1]
cumulative_share = np.cumsum(sorted_revenues) / sorted_revenues.sum() * 100

n_customers = len(sorted_revenues)
top_10_percent = int(np.ceil(n_customers * 0.10))
top_20_percent = int(np.ceil(n_customers * 0.20))

print(f"Top 10% of customers ({top_10_percent}) generate {cumulative_share[top_10_percent - 1]:.1f}% of revenue")
print(f"Top 20% of customers ({top_20_percent}) generate {cumulative_share[top_20_percent - 1]:.1f}% of revenue")
print("Customers needed to reach 50% of revenue:", int(np.argmax(cumulative_share >= 50)) + 1)

In [ ]:
# Best and worst months, and yearly change
best = monthly_df.loc[monthly_df["revenue"].idxmax()]
worst = monthly_df.loc[monthly_df["revenue"].idxmin()]
print("Highest month:", best["year_month"], "->", best["revenue"])
print("Lowest month :", worst["year_month"], "->", worst["revenue"])

yearly = df_clean.groupby(df_clean["OrderDate"].dt.year)["Revenue"].sum().round(2)
print()
print(yearly)
growth = (yearly[2024] / yearly[2023] - 1) * 100
print(f"\nRevenue change 2023 -> 2024: {growth:.1f}%")

is_nov_dec = df_clean["OrderDate"].dt.month.isin([11, 12])
nov_dec = df_clean[is_nov_dec].groupby(df_clean["OrderDate"].dt.year)["Revenue"].sum()
print("\nNov + Dec as % of the year's revenue:")
print((nov_dec / yearly * 100).round(1))

In [ ]:
# Average order value by country and by month
country_df["avg_order_value"] = (country_df["revenue"] / country_df["orders"]).round(2)
monthly_df["avg_order_value"] = (monthly_df["revenue"] / monthly_df["orders"]).round(2)
country_df

In [ ]:
# Extra checks used in the insights section
top10_share = top_products_df["revenue"].sum() / df_clean["Revenue"].sum() * 100
print(f"Top 10 products: {top10_share:.1f}% of revenue")

monthly_df["month_number"] = monthly_df["year_month"].str[5:].astype(int)
monthly_df["is_peak"] = monthly_df["month_number"].isin([11, 12])
print(monthly_df.groupby("is_peak")[["orders", "avg_order_value"]].mean().round(1))

year = df_clean["OrderDate"].dt.year
yearly_orders = df_clean.groupby(year)["OrderID"].nunique()
yearly_revenue = df_clean.groupby(year)["Revenue"].sum()
print(yearly_orders)
print((yearly_revenue / yearly_orders).round(2))

customers_2024 = df_clean[year == 2024]["CustomerID"].unique()
lapsed_df = customer_df[~customer_df["customer_id"].isin(customers_2024)]
print("Customers with no 2024 orders:", len(lapsed_df))
print("Their total revenue:", round(lapsed_df["revenue"].sum(), 2))
print("Share of all revenue:", round(lapsed_df["revenue"].sum() / df_clean["Revenue"].sum() * 100, 1), "%")
print("Their average revenue:", round(lapsed_df["revenue"].mean(), 2))

## 7. Visualizations
Seven charts, each chosen for a specific question. They are saved as PNG files in the `visualizations/` folder.

In [ ]:
Path("../visualizations").mkdir(exist_ok=True)

plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

with_commas = FuncFormatter(lambda value, position: f"{value:,.0f}")

def save_chart(filename):
    """Save the current chart as a PNG file and show it."""
    plt.tight_layout()
    plt.savefig(f"../visualizations/{filename}", dpi=150, bbox_inches="tight")
    plt.show()

### Chart 1: Monthly revenue (line chart: a value over time)

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(monthly_df["year_month"], monthly_df["revenue"], marker="o", color="#1f77b4")
plt.title("Monthly Revenue, 2023-2024")
plt.xlabel("Month")
plt.ylabel("Revenue")
plt.xticks(rotation=45, ha="right")
plt.gca().yaxis.set_major_formatter(with_commas)
plt.grid(axis="y", alpha=0.3)
save_chart("01_monthly_revenue.png")

### Chart 2: Revenue by category (bar chart: comparing groups)

In [ ]:
plt.figure(figsize=(8, 5))
bars = plt.bar(category_df["category"], category_df["revenue"], color="#1f77b4")
plt.bar_label(bars, fmt="{:,.0f}", padding=3)
plt.title("Revenue by Product Category")
plt.xlabel("Category")
plt.ylabel("Revenue")
plt.xticks(rotation=20, ha="right")
plt.gca().yaxis.set_major_formatter(with_commas)
save_chart("02_revenue_by_category.png")

### Chart 3: Top 10 products (horizontal bar chart: long product names)

In [ ]:
plt.figure(figsize=(9, 5.5))
bars = plt.barh(top_products_df["product_name"], top_products_df["revenue"], color="#1f77b4")
plt.bar_label(bars, fmt="{:,.0f}", padding=3)
plt.gca().invert_yaxis()
plt.title("Top 10 Products by Revenue")
plt.xlabel("Revenue")
plt.ylabel("Product")
plt.gca().xaxis.set_major_formatter(with_commas)
plt.xlim(0, top_products_df["revenue"].max() * 1.12)
save_chart("03_top_10_products.png")

### Chart 4: Orders per month (bar chart: a count per month)

In [ ]:
plt.figure(figsize=(10, 5))
plt.bar(monthly_df["year_month"], monthly_df["orders"], color="#1f77b4")
plt.title("Number of Orders per Month")
plt.xlabel("Month")
plt.ylabel("Orders")
plt.xticks(rotation=45, ha="right")
plt.grid(axis="y", alpha=0.3)
plt.gca().set_axisbelow(True)
save_chart("04_orders_by_month.png")

### Chart 5: Revenue by country (bar chart: comparing groups)

In [ ]:
known_countries = country_df[country_df["country"] != "Unknown"]

plt.figure(figsize=(9, 5))
bars = plt.bar(known_countries["country"], known_countries["revenue"], color="#1f77b4")
plt.bar_label(bars, fmt="{:,.0f}", padding=3)
plt.title("Revenue by Country")
plt.xlabel("Country")
plt.ylabel("Revenue")
plt.xticks(rotation=30, ha="right")
plt.gca().yaxis.set_major_formatter(with_commas)
plt.figtext(0.01, -0.03, "Excludes 1 order with unknown country (revenue 11.59).", fontsize=8, color="gray")
save_chart("05_revenue_by_country.png")

### Chart 6: Average order value per month (line chart with overall average)

In [ ]:
overall_aov = df_clean["Revenue"].sum() / df_clean["OrderID"].nunique()

plt.figure(figsize=(10, 5))
plt.plot(monthly_df["year_month"], monthly_df["avg_order_value"],
         marker="o", color="#1f77b4", label="Monthly average order value")
plt.axhline(overall_aov, color="gray", linestyle="--",
            label=f"Overall average ({overall_aov:,.0f})")
plt.title("Average Order Value per Month")
plt.xlabel("Month")
plt.ylabel("Average order value")
plt.xticks(rotation=45, ha="right")
plt.gca().yaxis.set_major_formatter(with_commas)
plt.grid(axis="y", alpha=0.3)
plt.legend()
save_chart("06_average_order_value.png")

### Chart 7: Distribution of revenue per customer (histogram: how values are spread)

In [ ]:
revenues = customer_df["revenue"].to_numpy()

plt.figure(figsize=(9, 5))
plt.hist(revenues, bins=30, color="#1f77b4", edgecolor="white")
plt.axvline(np.mean(revenues), color="black", linestyle="--",
            label=f"Mean ({np.mean(revenues):,.0f})")
plt.axvline(np.median(revenues), color="orange", linestyle="--",
            label=f"Median ({np.median(revenues):,.0f})")
plt.title("Distribution of Revenue per Customer")
plt.xlabel("Total revenue per customer")
plt.ylabel("Number of customers")
plt.gca().xaxis.set_major_formatter(with_commas)
plt.legend()
save_chart("07_customer_revenue_distribution.png")

## 8. Key Business Insights

*All findings describe this synthetic dataset (Jan 2023 - Dec 2024), not a real business.*

1. **Overall size.** The data contains 4,498 orders from 427 customers across 60 products: total revenue 2,884,150.03, 30,537 units sold, and an average order value of 641.21. *(SQL 6, KPI table)*
2. **Electronics dominates revenue.** Electronics generated 54.6% of revenue (1,574,765.51) from only 19.0% of units sold. Books generated 5.3% of revenue from 23.2% of units. *(SQL 8, Chart 2)*
3. **A few products carry the revenue.** 9 of the top 10 products by revenue are Electronics, and the top 10 products together make up 54.5% of revenue. Webcam HD is first with 230,428.58. *(SQL 9, Chart 3)*
4. **November and December are the peak.** The highest month is 2024-12 (219,856.88) and the lowest is 2023-09 (89,946.75). November and December made up 24.8% of 2023 revenue and 25.6% of 2024 revenue. *(SQL 10, Charts 1 and 4)*
5. **The peak comes from more orders, not larger ones.** November-December months averaged 288.8 orders versus 167.2 in other months, while the average order value was 628.1 versus 647.9. *(Charts 4 and 6)*
6. **Growth came from order count.** Revenue grew 16.6% from 2023 to 2024 (1,331,605.47 to 1,552,544.56) and orders grew from 2,048 to 2,450, while the yearly average order value moved from 650.20 to 633.69. *(Step 10, Cell 5)*
7. **Two countries bring half of revenue.** The United Kingdom (781,957.01) and the United States (717,804.84) together account for 52.0% of revenue. Pakistan is fourth with 10.5%. Average order value ranges from 559.70 (Canada) to 687.45 (Australia, which has the fewest orders: 178). *(SQL 11, Chart 5)*
8. **Revenue depends on a minority of customers.** 93 customers (21.8% of all customers) with at least 10,000 in total spending generate 57.9% of revenue. The top 10% of customers (43) generate 35.5%. The average customer spent 6,754.45 while the median customer spent 4,558.54. *(SQL 18, Step 10, Chart 7)*
9. **Premium products earn most revenue.** The 19 products priced at 100 or more generated 68.7% of revenue from 30.3% of units. The 19 products under 30 generated 7.5% of revenue from 33.4% of units. *(SQL 14)*
10. **Some high-volume products earn little revenue.** Mystery Novel (943 units, rank 2 by units but rank 45 by revenue), Cooking Basics (884 units, rank 4 by units, rank 41 by revenue) and Electric Kettle (712 units, rank 14 by units, rank 46 by revenue) are in the top 20 for units but in the bottom 20 for revenue. *(SQL 19)*
11. **Customers who stopped buying are small.** 33 customers (7.7%) placed no order in 2024. Together they generated 32,143.16, which is 1.1% of revenue, and their average spending was 974.04. *(SQL 16, Step 13 check)*

## Business Recommendations

*Each recommendation is based on the finding number shown. They are suggestions for a real business, not proven results.*

| # | Data finding | Recommendation |
|---|---|---|
| 1 | Findings 2, 3: Electronics is 54.6% of revenue and 9 of the top 10 products. | Keep the top Electronics products well stocked and monitor their share over time, because the business depends heavily on them. |
| 2 | Findings 4, 5: Nov-Dec months had about 1.7 times the orders of other months. | Plan stock, staffing and marketing for November and December ahead of time. |
| 3 | Finding 6: growth came from more orders while average order value slightly decreased. | Test offers that raise the order value, such as bundles, and compare the result with the current 641.21. |
| 4 | Finding 8: 93 customers generate 57.9% of revenue. | Track the order frequency of high-value customers and consider a retention programme for them. |
| 5 | Finding 11: customers who stopped buying generated only 1.1% of revenue. | Give win-back campaigns a lower priority than retaining Medium and High customers. |
| 6 | Findings 2, 10: Books and three products sell many units but earn little revenue. | Review pricing and bundling for Mystery Novel, Cooking Basics and Electric Kettle. There is no cost data here, so profit cannot be judged. |
| 7 | Finding 7: two countries give 52.0% of revenue. | Monitor dependence on the UK and US. The differences in average order value between countries are small, so they should be tested before acting on them. |

## Limitations

- The dataset is synthetic, so patterns reflect how it was generated.
- There is no cost or profit data, so the analysis covers revenue only.
- The data covers 2 years and has no returns or marketing information, so causes cannot be proven.